# Hull Tactical - Market Prediction

## 元notebook: Hull Starter Notebook

- **原著者**: Laurent Lanteigne（[laurentlanteigne](https://www.kaggle.com/laurentlanteigne)）ほか
- **元notebookへのリンク**: https://www.kaggle.com/code/laurentlanteigne/hull-starter-notebook
- **評価**: 1738 upvotes / Gold medal（コンペのCodeタブにピン留めされている公式スターターノートブック）

### 概要

過去の匿名化された市場特徴量（`D`, `E`, `I`, `M`, `P`, `S`, `U` などから始まる列名）を使って、翌日の「市場の超過リターン（forward excess return）」を予測し、それを 0〜2 の範囲のポジションサイズ（取引シグナル）に変換する、金融市場予測のCode Competition。このnotebookは、単なるモデリングだけでなく、**設定値の一元管理（dataclass）→データ読み込み→特徴量作成→学習→Kaggle評価サーバー経由での推論提供**という、実務に近いパイプライン設計の型を学べる点が特徴。

### 断り書き

これは学習目的の解説付き写しです。元のコードセルの中身は変更していませんが、このノートブックは未実行のため、グラフや表などの出力は含まれていません。

## ライブラリの読み込み

**What**: `polars`（高速なDataFrameライブラリ）、`numpy`、`scikit-learn`（ElasticNet系のモデルとスケーラー）、そして `kaggle_evaluation`（Kaggleが提供する評価用サーバーのライブラリ）をインポートする。

**Why**: このコンペは「Code Competition」と呼ばれる形式で、モデルを学習するだけでなく、`kaggle_evaluation` が要求する形式で予測関数を実装し、評価サーバーに接続する必要がある。`pandas` ではなく `polars` を使っているのは、大きめの時系列データを高速に処理するため。

In [ ]:
import os
from pathlib import Path
import datetime

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl
import numpy as np
from sklearn.linear_model import ElasticNet, ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler

import kaggle_evaluation.default_inference_server

## プロジェクト（入力データ）のディレクトリ構造を確認

**What**: `/kaggle/input` 以下をたどって、利用できるファイルの一覧を出力する。

**Why**: このコンペでは `train.csv` / `test.csv` に加えて、提出のために使う `kaggle_evaluation` 関連のPythonファイル・protoファイルが用意されている。最初にどんなファイルがあるかを把握しておくことで、この後のコードで参照するパスの見通しが立つ。

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 設定値（Configurations）の一元管理

**What**: データのパス、リターン→シグナル変換用の上限・下限・倍率、モデルのハイパーパラメータ（交差検証の分割数、ElasticNetの `l1_ratio`、正則化強度の候補 `alphas`、最大反復回数）を、コードの先頭でまとめて定数として定義する。

**Why**: パラメータをコードのあちこちに直書きすると、後で調整するときに探し回ることになる。設定値を一箇所にまとめておくことで、実験の再現性が上がり、パラメータチューニングもしやすくなる。これは実務のMLコードでもよく使われる書き方。

In [ ]:
# ============= PATHS =============
DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')

# ============= RETURNS TO SIGNAL CONFIGS =============
MIN_SIGNAL: float = 0.0                    # Minimum value for the daily signal
MAX_SIGNAL: float = 2.0                    # Maximum value for the daily signal
SIGNAL_MULTIPLIER: float = 400.0           # Multiplier of the OLS market forward excess returns predictions to signal

# ============= MODEL CONFIGS =============
CV: int = 10                               # Number of cross validation folds in the model fitting
L1_RATIO: float = 0.5                      # ElasticNet mixing parameter
ALPHAS: np.ndarray = np.logspace(-4, 2, 100)  # Constant that multiplies the penalty terms
MAX_ITER: int = 1000000                    # The maximum number of iterations

## データクラス（dataclass）によるヘルパー定義

**What**: Pythonの `@dataclass` を使って、3つの入れ物を定義する。

- `DatasetOutput`: 分割・スケーリング済みの学習/検証データ一式（`X_train`, `X_test`, `y_train`, `y_test`, `scaler`）をまとめて持ち運ぶための箱
- `ElasticNetParameters`: ElasticNetのハイパーパラメータ一式。`__post_init__` で `l1_ratio` が0〜1の範囲外なら例外を出すバリデーションが入っている
- `RetToSignalParameters`（`frozen=True`）: リターンをシグナルに変換する際のパラメータ。`frozen=True` にすることで、一度作ったら値を書き換えられないようにしている

**Why**: 関数の引数や戻り値が増えてくると、タプルや辞書で受け渡すよりも `dataclass` にした方が「何が入っているか」が名前から分かり、型ヒントも効くのでバグを減らせる。`frozen=True` は「設定値は途中で誤って変更されたくない」という意図を明示する書き方。

In [ ]:
@dataclass
class DatasetOutput:
    X_train : pl.DataFrame
    X_test: pl.DataFrame
    y_train: pl.Series
    y_test: pl.Series
    scaler: StandardScaler

@dataclass
class ElasticNetParameters:
    l1_ratio : float
    cv: int
    alphas: np.ndarray
    max_iter: int

    def __post_init__(self):
        if self.l1_ratio < 0 or self.l1_ratio > 1:
            raise ValueError("Wrong initializing value for ElasticNet l1_ratio")

@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float
    min_signal : float = MIN_SIGNAL
    max_signal: float = MAX_SIGNAL

## パラメータオブジェクトの生成

**What**: 先ほど定義した `RetToSignalParameters` と `ElasticNetParameters` に、実際の設定値（定数）を渡してインスタンスを作る。

**Why**: ここで一度オブジェクト化しておくことで、この先の関数にはこの2つのオブジェクトを渡すだけでよくなり、関数の引数がすっきりする。

In [ ]:
ret_signal_params = RetToSignalParameters(
    signal_multiplier= SIGNAL_MULTIPLIER
)

enet_params = ElasticNetParameters(
    l1_ratio = L1_RATIO,
    cv = CV,
    alphas = ALPHAS,
    max_iter = MAX_ITER
)

## データ読み込み・特徴量作成のヘルパー関数群

**What**: 5つの関数をまとめて定義する。

- `load_trainset()`: `train.csv` を読み込み、列名 `market_forward_excess_returns` を `target` にリネームし、末尾10行を除いてDataFrameとして返す
- `load_testset()`: `test.csv` を読み込み、同様に `lagged_forward_returns` を `target` にリネームする
- `create_example_dataset()`: 使う特徴量を絞り込み（`vars_to_keep`）、いくつかの列同士の差や比率から新しい特徴量（`U1`, `U2`）を作り、欠損値を指数加重移動平均で埋めてから、欠損の残る行を削除する
- `join_train_test_dataframes()`: 学習・テストの共通列だけを取り出して縦に連結する（後段の特徴量エンジニアリングを両方に一括で適用するための下準備）
- `split_dataset()`: 特徴量とターゲットに分割し、`StandardScaler` で標準化（平均0・分散1に変換）した上で `DatasetOutput` にまとめて返す

**Why**: 「読み込み」「特徴量作成」「学習/テストへの分割」という一連の処理を関数化しておくことで、後段のコードが読みやすくなり、同じ前処理を学習データにもテストデータにも一貫して適用できる（これを崩すと学習時とテスト時で処理が食い違う「データリーク」やバグの温床になる）。標準化を行うのは、ElasticNetのような線形モデルは特徴量のスケールの影響を受けやすいため。

In [ ]:
def load_trainset() -> pl.DataFrame:
    """
    Loads and preprocesses the training dataset.

    Returns:
        pl.DataFrame: The preprocessed training DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "train.csv")
        .rename({'market_forward_excess_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
        .head(-10)
    )

def load_testset() -> pl.DataFrame:
    """
    Loads and preprocesses the testing dataset.

    Returns:
        pl.DataFrame: The preprocessed testing DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
    )

def create_example_dataset(df: pl.DataFrame) -> pl.DataFrame:
    """
    Creates new features and cleans a DataFrame.

    Args:
        df (pl.DataFrame): The input Polars DataFrame.

    Returns:
        pl.DataFrame: The DataFrame with new features, selected columns, and no null values.
    """
    vars_to_keep: List[str] = [
        "S2", "E2", "E3", "P9", "S1", "S5", "I2", "P8",
        "P10", "P12", "P13", "U1", "U2"
    ]

    return (
        df.with_columns(
            (pl.col("I2") - pl.col("I1")).alias("U1"),
            (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
        )
        .select(["date_id", "target"] + vars_to_keep)
        .with_columns([
            pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5))
            for col in vars_to_keep
        ])
        .drop_nulls()
    )

def join_train_test_dataframes(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    """
    Joins two dataframes by common columns and concatenates them vertically.

    Args:
        train (pl.DataFrame): The training DataFrame.
        test (pl.DataFrame): The testing DataFrame.

    Returns:
        pl.DataFrame: A single DataFrame with vertically stacked data from common columns.
    """
    common_columns: list[str] = [col for col in train.columns if col in test.columns]

    return pl.concat([train.select(common_columns), test.select(common_columns)], how="vertical")

def split_dataset(train: pl.DataFrame, test: pl.DataFrame, features: list[str]) -> DatasetOutput:
    """
    Splits the data into features (X) and target (y), and scales the features.

    Args:
        train (pl.DataFrame): The processed training DataFrame.
        test (pl.DataFrame): The processed testing DataFrame.
        features (list[str]): List of features to used in model.

    Returns:
        DatasetOutput: A dataclass containing the scaled feature sets, target series, and the fitted scaler.
    """
    X_train = train.drop(['date_id','target'])
    y_train = train.get_column('target')
    X_test = test.drop(['date_id','target'])
    y_test = test.get_column('target')

    scaler = StandardScaler()

    X_train_scaled_np = scaler.fit_transform(X_train)
    X_train = pl.from_numpy(X_train_scaled_np, schema=features)

    X_test_scaled_np = scaler.transform(X_test)
    X_test = pl.from_numpy(X_test_scaled_np, schema=features)

    return DatasetOutput(
        X_train = X_train,
        y_train = y_train,
        X_test = X_test,
        y_test = y_test,
        scaler = scaler
    )

## 予測リターンを取引シグナルに変換する関数

**What**: モデルが予測した「期待リターン」の配列を受け取り、`signal_multiplier` を掛けたうえで `min_signal`〜`max_signal`（0〜2）の範囲にクリップ（切り詰め）して返す関数。

**Why**: モデルの生の予測値（リターン）をそのまま取引量として使うと、値が極端になったときにポジションが過大・過小になりすぎる恐れがある。`np.clip` で範囲を制限することで、予測が外れた場合のリスクを一定の範囲に抑える、という金融特有のリスク管理の考え方が反映されている。

In [ ]:
def convert_ret_to_signal(
    ret_arr: np.ndarray,
    params: RetToSignalParameters
) -> np.ndarray:
    """
    Converts raw model predictions (expected returns) into a trading signal.

    Args:
        ret_arr (np.ndarray): The array of predicted returns.
        params (RetToSignalParameters): Parameters for scaling and clipping the signal.

    Returns:
        np.ndarray: The resulting trading signal, clipped between min and max values.
    """
    return np.clip(
        ret_arr * params.signal_multiplier + 1, params.min_signal, params.max_signal
    )

## データの中身を確認

**What**: 先に定義した `load_trainset()` / `load_testset()` を実行し、学習データの末尾3行とテストデータの先頭3行を表示する。

**Why**: 関数を作った後は、実際にデータを読み込んで列名やデータ型、行数が想定通りかを目視確認するのが安全。とくに `train` と `test` で列構成が違う（`test` には `is_scored` や `lagged_risk_free_rate` などが含まれる）ことにここで気づける。

In [ ]:
train: pl.DataFrame = load_trainset()
test: pl.DataFrame = load_testset()
print(train.tail(3))
print(test.head(3))

## 学習データ・テストデータの生成（特徴量エンジニアリングの適用）

**What**: 学習・テストを一旦連結して `create_example_dataset()`（特徴量作成）を通した後、元の `date_id` を使って再び学習用・テスト用に分割し直す。そこから使用する特徴量の一覧 `FEATURES` を作り、`split_dataset()` でスケーリング済みのX/yに変換する。

**Why**: 特徴量エンジニアリング（列の差分や比率を取る、欠損を埋めるなど）を学習データとテストデータに別々に書くと処理がずれるリスクがある。一度連結してから同じ関数を通し、その後で `date_id` を使って再分割する、という書き方によって「学習時とテスト時で同じ前処理をしている」ことを保証している。

In [ ]:
df: pl.DataFrame = join_train_test_dataframes(train, test)
df = create_example_dataset(df=df)
train: pl.DataFrame = df.filter(pl.col('date_id').is_in(train.get_column('date_id')))
test: pl.DataFrame = df.filter(pl.col('date_id').is_in(test.get_column('date_id')))

FEATURES: list[str] = [col for col in test.columns if col not in ['date_id', 'target']]

dataset: DatasetOutput = split_dataset(train=train, test=test, features=FEATURES)

X_train: pl.DataFrame = dataset.X_train
X_test: pl.DataFrame = dataset.X_test
y_train: pl.DataFrame = dataset.y_train
y_test: pl.DataFrame = dataset.y_test
scaler: StandardScaler = dataset.scaler

## モデルの学習（ElasticNetCV → ElasticNet）

**What**: まず `ElasticNetCV` で、あらかじめ用意した `alphas`（正則化強度の候補）の中から交差検証（`cv=10`）によって最適な `alpha` を探索する。そのあと、見つかった最適な `alpha` を使って本番用の `ElasticNet` モデルを学習し直す。

**Why**: ElasticNetはRidge回帰（L2正則化）とLasso回帰（L1正則化）を組み合わせた線形モデルで、`l1_ratio` でその配合を決める。正則化の強さ（`alpha`）を人手で決め打ちすると過学習・未学習のどちらかに偏りやすいため、交差検証で自動的に適切な強さを選ぶのが定石。市場データのようにノイズが多く特徴量同士の相関が高いデータでは、ElasticNetのような正則化付き線形モデルが安定した第一歩の選択肢になりやすい。

In [ ]:
model_cv: ElasticNetCV = ElasticNetCV(
    **asdict(enet_params)
)
model_cv.fit(X_train, y_train)

# Fit the final model using the best alpha found by cross-validation
model: ElasticNet = ElasticNet(alpha=model_cv.alpha_, l1_ratio=enet_params.l1_ratio)
model.fit(X_train, y_train)

## Kaggle評価サーバー用の予測関数

**What**: このコンペの提出形式（Code Competition）が要求する `predict(test)` 関数を定義する。渡された1行分のテストデータに対して、列名のリネーム→特徴量作成→スケーリング→モデル予測→シグナル変換、という一連の処理を行い、最終的な取引シグナル（1つの `float`）を返す。

**Why**: Code Competitionでは、学習済みモデルをそのまま提出するのではなく、「新しいデータが来るたびに1件ずつ予測を返す関数」を評価サーバーに登録する必要がある。ここまでに作った前処理関数（`create_example_dataset`）とスケーラー（`scaler`）、学習済み `model`、そして `convert_ret_to_signal` を全部つなげているのがこの関数。

In [ ]:
def predict(test: pl.DataFrame) -> float:
    test = test.rename({'lagged_forward_returns':'target'})
    df: pl.DataFrame = create_example_dataset(test)
    X_test: pl.DataFrame = df.select(FEATURES)
    X_test_scaled_np: np.ndarray = scaler.transform(X_test)
    X_test: pl.DataFrame = pl.from_numpy(X_test_scaled_np, schema=FEATURES)
    raw_pred: float = model.predict(X_test)[0]
    return convert_ret_to_signal(raw_pred, ret_signal_params)

## 推論サーバーの起動

**What**: `kaggle_evaluation.default_inference_server.DefaultInferenceServer` に、先ほど定義した `predict` 関数を登録する。環境変数 `KAGGLE_IS_COMPETITION_RERUN` が立っている（＝本番のコンペ再実行環境で動いている）場合は `serve()` で実際にサーバーとして待ち受け、そうでない場合（手元での動作確認）は `run_local_gateway()` でローカルにあるテストデータを使って動作をシミュレーションする。

**Why**: Code Competitionでは、提出したnotebookが本番環境で自動的に再実行され、この推論サーバーを通じて1件ずつ判定される。ローカルでも同じコードで動作確認ができるようになっている点が、この分岐の工夫。

In [ ]:
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

## まとめ

このnotebookは、金融市場の時系列データに対する「設定管理→特徴量エンジニアリング→交差検証つき線形モデル→提出用の推論サーバー実装」という一連の流れをコンパクトに示した、Hull Tacticalコンペの公式スターターノートブック。dataclassによる設定・データの型付けや、学習時とテスト時で同じ前処理関数を共有する設計は、コンペに限らず実務のMLパイプラインでも役立つ書き方。

このノートブックはApache 2.0ライセンスの下で公開されている。